# Svara TTS API Demo

This notebook demonstrates the Intel Text-to-Speech API with 88 multi-language voices.

## System Status
- ✅ TTS API Pod: snac-decoder (Kubernetes)
- ✅ Service Endpoint: NodePort (External usage)
- ✅ VLLM Backend: Connected
- ✅ SNAC Codec: Loaded
- ✅ Available Voices: 88

## Setup
Import required libraries

In [1]:
import requests
import json
from IPython.display import Audio, display
import pandas as pd
import os

def get_service_endpoint(service_name="snac-decoder-service", namespace="default", port=30800, node_ip="172.31.51.24"):
    """
    Get the endpoint for a Kubernetes service.
    
    Priority:
    1. Environment variable TTS_API_URL (for manual override)
    2. NodePort Access (External Access via Node IP)
    3. Kubernetes DNS (automatic, no API calls needed)
    """
    # 1. Check environment variable first
    if os.environ.get("TTS_API_URL"):
        return os.environ.get("TTS_API_URL")
    
    # 2. Try NodePort Access (External)
    # For NodePort, we use the Node's IP (or localhost if port-forwarded/local) and the NodePort
    nodeport_endpoint = f"http://{node_ip}:{port}"
    
    try:
        response = requests.get(f"{nodeport_endpoint}/health", timeout=2)
        if response.status_code == 200:
            return nodeport_endpoint
    except:
        pass
    
    # 3. Fallback to K8s DNS (Inside Cluster)
    dns_endpoint = f"http://{service_name}.{namespace}.svc.cluster.local:8000"
    return dns_endpoint

# TTS API Configuration
# Update node_ip to your actual K8s node IP if not running on the same machine
# We use the NodePort 30800 and the Node IP discovered from the cluster
BASE_URL = get_service_endpoint(port=8000, node_ip="35.86.73.253") 

print(f"✅ Connected to TTS API at: {BASE_URL}")

✅ Connected to TTS API at: http://snac-decoder-service.default.svc.cluster.local:8000


## 1. Health Check
Verify the TTS API server is healthy and connected to all services

In [2]:
try:
    response = requests.get(f"{BASE_URL}/health", timeout=5)
    health = response.json()
    
    print("🏥 Health Check Results:")
    print(f"  Status: {health['status'].upper()}")
    print(f"  SNAC Codec: {'✅ Loaded' if health['snac_loaded'] else '❌ Not Loaded'}")
    print(f"  VLLM Backend: {'✅ Connected' if health['vllm_connected'] else '❌ Disconnected'}")
    print(f"  Available Voices: {health['voices_loaded']}")
    
except Exception as e:
    print(f"❌ Health check failed: {e}")

🏥 Health Check Results:
  Status: HEALTHY
  SNAC Codec: ✅ Loaded
  VLLM Backend: ✅ Connected
  Available Voices: 88


## 2. List Available Voices
Explore the 88 voices across multiple languages

In [3]:
try:
    response = requests.get(f"{BASE_URL}/v1/voices")
    voices = response.json()
    
    print(f"📢 Found {len(voices)} voices\n")
    
    # Create a DataFrame for better visualization
    df = pd.DataFrame(voices)
    
    # Show language distribution
    print("🌍 Languages Available:")
    print(df['language'].value_counts().head(10))
    
    print("\n📝 Sample Voices:")
    display(df[['id', 'name', 'language', 'gender']].head(15))
    
except Exception as e:
    print(f"❌ Failed to list voices: {e}")

📢 Found 88 voices

🌍 Languages Available:
language
hi     52
bn      2
mr      2
te      2
kn      2
bh      2
mag     2
hne     2
mai     2
as      2
Name: count, dtype: int64

📝 Sample Voices:


,id,name,language,gender
0,hi_male,Hindi (Male),hi,male
1,hi_female,Hindi (Female),hi,female
2,bn_male,Bengali (Male),bn,male
3,bn_female,Bengali (Female),bn,female
4,mr_male,Marathi (Male),mr,male
5,mr_female,Marathi (Female),mr,female
6,te_male,Telugu (Male),te,male
7,te_female,Telugu (Female),te,female
8,kn_male,Kannada (Male),kn,male
9,kn_female,Kannada (Female),kn,female


## 3. Generate Speech (Hindi Male)
Generate audio using a Hindi male voice

In [4]:
payload = {
    "text": "नमस्ते, आज का दिन बहुत अच्छा है।",  # Shorter Hindi text for testing
    "voice": "hi_male",
    "format": "wav",
    "speed": 1.0
}

print("🎤 Generating audio with Hindi male voice...")
print("⏱️  Note: CPU generation is slow (~1-2 tokens/sec), this may take several minutes...")

try:
    response = requests.post(f"{BASE_URL}/v1/text-to-speech", json=payload, timeout=600)  # Increased to 10 minutes
    
    if response.status_code == 200:
        print("✅ Audio generated successfully!")
        print(f"📊 Audio size: {len(response.content):,} bytes")
        
        # Play audio in notebook
        display(Audio(response.content, autoplay=False, rate=24000))
        
        # Save to file
        with open("output_hindi_male.wav", "wb") as f:
            f.write(response.content)
        print("💾 Saved to: output_hindi_male.wav")
    else:
        print(f"❌ Error {response.status_code}: {response.text}")
        
except Exception as e:
    print(f"❌ TTS generation failed: {e}")

🎤 Generating audio with Hindi male voice...
⏱️  Note: CPU generation is slow (~1-2 tokens/sec), this may take several minutes...


✅ Audio generated successfully!
📊 Audio size: 131,116 bytes


💾 Saved to: output_hindi_male.wav


## 4. Generate Speech (English Female)
Try another voice

In [5]:
payload = {
    "text": "Hello world, this is a test. Pradeep",  # Shorter text for faster testing
    "voice": "en_female",
    "format": "wav",
    "speed": 1.0
}

print("🎤 Generating audio with English female voice...")
print("⏱️  Note: CPU generation is slow (~1-2 tokens/sec), this may take several minutes...")

try:
    response = requests.post(f"{BASE_URL}/v1/text-to-speech", json=payload, timeout=600)  # Increased to 10 minutes
    
    if response.status_code == 200:
        print("✅ Audio generated successfully!")
        print(f"📊 Audio size: {len(response.content):,} bytes")
        
        # Play audio in notebook
        display(Audio(response.content, autoplay=False, rate=24000))
        
        # Save to file
        with open("output_english_female.wav", "wb") as f:
            f.write(response.content)
        print("💾 Saved to: output_english_female.wav")
    else:
        print(f"❌ Error {response.status_code}: {response.text}")
        
except Exception as e:
    print(f"❌ TTS generation failed: {e}")

🎤 Generating audio with English female voice...
⏱️  Note: CPU generation is slow (~1-2 tokens/sec), this may take several minutes...


✅ Audio generated successfully!
📊 Audio size: 102,444 bytes


❌ TTS generation failed: [Errno 2] No such file or directory: 'output_english_female.wav'


## 5. Custom Voice Test
Try your own text and voice!

In [6]:
# Customize these values
custom_text = "Hello world.I'm Pradeep"  # Keep it very short for CPU testing
custom_voice = "bn_male"  # Try: hi_female, mr_male, en_male, etc.

payload = {
    "text": custom_text,
    "voice": custom_voice,
    "format": "wav",
    "speed": 1.0
}

print(f"🎤 Generating audio with voice: {custom_voice}...")
print(f"📝 Text: {custom_text}")
print("⏱️  Note: CPU generation is slow (~1-2 tokens/sec), this may take several minutes...")

try:
    response = requests.post(f"{BASE_URL}/v1/text-to-speech", json=payload, timeout=600)  # Increased to 10 minutes
    
    if response.status_code == 200:
        print("✅ Audio generated successfully!")
        display(Audio(response.content, autoplay=False, rate=24000))
        
        filename = f"output_{custom_voice}.wav"
        with open(filename, "wb") as f:
            f.write(response.content)
        print(f"💾 Saved to: {filename}")
    else:
        print(f"❌ Error {response.status_code}: {response.text}")
        
except Exception as e:
    print(f"❌ TTS generation failed: {e}")

🎤 Generating audio with voice: bn_male...
📝 Text: Hello world.I'm Pradeep
⏱️  Note: CPU generation is slow (~1-2 tokens/sec), this may take several minutes...


❌ TTS generation failed: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
